# BrownDye2: complex PQR preparation

Generate a PQR file and APBS electrostatic potential map for a protein-ligand complex.

**This notebook**: prepare protein and ligand inputs from a docked PDB.
1. Fix protein with PDBFixer (add missing atoms, protonate at target pH)
2. Assign ligand bond orders from SMILES, write SDF for antechamber

**`run_amber_apbs.sh`**: parameterize and solve electrostatics.
1. `pdb4amber` -- strip H/water, fix residue names for tleap
2. `antechamber` + `parmchk2` -- GAFF2 atom types and AM1-BCC charges
3. `tleap` -- combine protein + ligand into AMBER topology
4. `ParmEd` -- convert prmtop/rst7 to PQR (with mbondi3 radii)
5. `inputgen` -- generate APBS input from PQR dimensions
6. `APBS` -- solve the linearized Poisson-Boltzmann equation

In [ ]:
from pathlib import Path

from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import ChainSelect, assign_topology, fix_pdb

COMPLEX_PDB = Path("TurboID-bioAMP_model_0.pdb")
WORKDIR = Path("tmp")
WORKDIR.mkdir(exist_ok=True)

# Canonical SMILES of the ligand (used to assign bond orders)
LIGAND_SMILES = r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)[C@@H](O)[C@H]3O"
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

## Step 1: Fix protein

Extract protein chain and add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract and fix protein chain
protein_pdb = WORKDIR / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

protein_fixed_pdb = WORKDIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract ligand chain, assign bond orders from a SMILES template, and write an SDF
for antechamber. The SMILES is needed because PDB files lack bond-order information.

In [ ]:
# Extract ligand chain to PDB for RDKit parsing
ligand_pdb = WORKDIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from SMILES template
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Write SDF for antechamber
lig_resnames = {res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()}
mol_assigned.SetProp("_Name", next(iter(lig_resnames)))

ligand_sdf = WORKDIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)
print(f"Ligand SDF -> {ligand_sdf}")

## Next: run_amber_apbs.sh

The notebook produced `tmp/protein_fixed.pdb` and `tmp/ligand.sdf`. The shell script
picks up from here -- it loads the protein and ligand **separately** into tleap
(via `pdb4amber` and `antechamber`), combines them, and runs APBS.

```bash
conda activate ambertools
cd examples/browndye && bash run_amber_apbs.sh
```

Outputs in `tmp/`:

| File | Description |
|------|-------------|
| `complex.prmtop` | AMBER topology |
| `complex.rst7` | AMBER coordinates |
| `complex.pqr` | PQR with AM1-BCC charges and mbondi3 radii |
| `complex.in` | APBS input (mg-auto, LPBE) |
| `complex.dx` | Electrostatic potential map (OpenDX) |